# In-Class Coding — Diffusion Models
### Lec8 · Spring 2026

This notebook follows the same style as Lec7 ICCP. We will implement and test three pieces:

1. **Forward process:** construct $q(x_t\mid x_0)$ and visualize noising.
2. **Backward process by predicting $x_0$:** train a clean-image denoiser and use it for reverse updates.
3. **Backward process by predicting noise:** train $\varepsilon_\theta(x_t,t)$ and compare deterministic vs. posterior DDPM sampling.

Work through each problem in order. Each problem has a test cell immediately after it.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Small MNIST subset for in-class runtime.
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2.0 - 1.0),  # map [0,1] -> [-1,1]
])
train_ds_full = datasets.MNIST(root='./data', train=True, download=True, transform=tfm)
test_ds_full  = datasets.MNIST(root='./data', train=False, download=True, transform=tfm)

train_ds = Subset(train_ds_full, range(12000))
test_ds  = Subset(test_ds_full, range(2000))
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=512, shuffle=False)
imgs_test, labels_test = next(iter(test_dl))

print(f'MNIST subset: train={len(train_ds):,} test={len(test_ds):,}')

def to_display(x):
    return ((x.detach().cpu().clamp(-1, 1) + 1.0) / 2.0)

def show_grid(images, title='', n=10):
    images = images.detach().cpu() if torch.is_tensor(images) else torch.tensor(images)
    images = images[:n]
    fig, axes = plt.subplots(1, n, figsize=(1.3*n, 1.5))
    for i, ax in enumerate(axes):
        ax.imshow(to_display(images[i:i+1])[0, 0], cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout(); plt.show()

print('Ready.')


---
# Section 0 — Forward Diffusion Process

The forward process is fixed and known:

$$x_t = \sqrt{\bar\alpha_t}x_0 + \sqrt{1-\bar\alpha_t}\varepsilon,\qquad \varepsilon\sim\mathcal{N}(0,I).$$

In this section, you implement the schedule and the closed-form noising function.


## Problem 1 — Build the Forward Schedule and Sample $x_t$

Your tasks:
1. Implement `linear_beta_schedule`.
2. Implement `extract`, which gathers per-timestep coefficients for a batch.
3. Implement `q_sample(x0, t, noise=None)`.
4. Implement `predict_x0_from_eps`, the exact inverse when the true noise is known.


In [ ]:
# -- Problem 1: forward schedule and q(x_t | x0) -----------------------------

T = 200

def linear_beta_schedule(T, beta_start=1e-4, beta_end=8e-2):
    """Return a length-T tensor of linearly spaced beta values."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement linear_beta_schedule')
    # -----------------------------------------------------------------------

betas = linear_beta_schedule(T).to(device)        # beta_1,...,beta_T
alphas = 1.0 - betas                              # alpha_t = 1 - beta_t
alpha_bar = torch.cumprod(alphas, dim=0)          # alpha_bar_t for t=1..T
alpha_bar0 = torch.cat([torch.ones(1, device=device), alpha_bar], dim=0)  # t=0..T


def extract(a, t, x_shape):
    """Gather a[t] for each item in the batch and reshape for broadcasting."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement extract')
    # -----------------------------------------------------------------------


def q_sample(x0, t, noise=None):
    """Sample x_t from q(x_t | x0), where t is an integer tensor in [0,T]."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement q_sample')
    # -----------------------------------------------------------------------


def predict_x0_from_eps(x_t, t, eps):
    """Recover x0 exactly from x_t if the true eps is known."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement predict_x0_from_eps')
    # -----------------------------------------------------------------------

print('Schedule ready. alpha_bar_T =', float(alpha_bar0[T]))


In [ ]:
# -- Test P1 -----------------------------------------------------------------
assert betas.shape == (T,), f'P1 FAIL: betas shape should be ({T},), got {betas.shape}'
assert torch.all((betas > 0) & (betas < 1)), 'P1 FAIL: all betas must be in (0,1)'
assert alpha_bar0.shape == (T + 1,), 'P1 FAIL: alpha_bar0 must have length T+1'
assert torch.isclose(alpha_bar0[0], torch.tensor(1.0, device=device)), 'P1 FAIL: alpha_bar0[0] should be 1'

x0 = imgs_test[:16].to(device)
t = torch.randint(1, T + 1, (x0.shape[0],), device=device)
eps = torch.randn_like(x0)
x_t = q_sample(x0, t, eps)
x0_rec = predict_x0_from_eps(x_t, t, eps)
assert x_t.shape == x0.shape, 'P1 FAIL: q_sample should preserve image shape'
assert F.mse_loss(x0_rec, x0).item() < 1e-10, 'P1 FAIL: true eps should reconstruct x0 exactly'
print('P1 PASSED')


## Problem 2 — Visualize the Forward Process

Implement `make_forward_row`, then compare the same digit at several timesteps.


In [ ]:
# -- Problem 2: forward visualization ---------------------------------------

def make_forward_row(x0_one, steps):
    """Return a list of x_t images for one image and a list of integer timesteps."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement make_forward_row')
    # -----------------------------------------------------------------------

steps = [0, 10, 50, 100, 150, T]
row = make_forward_row(imgs_test[0:1].to(device), steps)
show_grid(torch.cat(row, dim=0), title='Forward noising of one MNIST digit', n=len(steps))


In [ ]:
# -- Test P2 -----------------------------------------------------------------
steps_t = [0, 1, 10, T]
row_t = make_forward_row(imgs_test[1:2].to(device), steps_t)
assert isinstance(row_t, list) and len(row_t) == len(steps_t), 'P2 FAIL: return a list with one image per step'
assert all(tuple(x.shape) == (1, 1, 28, 28) for x in row_t), 'P2 FAIL: each item should have shape (1,1,28,28)'
print('P2 PASSED')


---
# Section 1 — Backward Process by Predicting $x_0$

Train a model $g_\theta(x_t,t)\approx x_0$. Then use the predicted clean image to move backward.


In [ ]:
# -- Shared tiny denoiser model ---------------------------------------------

def sinusoidal_time_embedding(t, dim=64):
    half = dim // 2
    freqs = torch.exp(torch.linspace(0, np.log(10000), half, device=t.device) * (-1))
    args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    emb = torch.cat([torch.sin(args), torch.cos(args)], dim=1)
    if dim % 2 == 1:
        emb = torch.cat([emb, torch.zeros_like(emb[:, :1])], dim=1)
    return emb

class TinyDenoiser(nn.Module):
    def __init__(self, out_tanh=False):
        super().__init__()
        self.time_mlp = nn.Sequential(nn.Linear(64, 128), nn.SiLU(), nn.Linear(128, 128))
        self.t1 = nn.Linear(128, 32)
        self.t2 = nn.Linear(128, 64)
        self.c1 = nn.Conv2d(1, 32, 3, padding=1)
        self.c2 = nn.Conv2d(32, 64, 3, padding=1)
        self.c3 = nn.Conv2d(64, 64, 3, padding=1)
        self.c4 = nn.Conv2d(64, 32, 3, padding=1)
        self.out = nn.Conv2d(32, 1, 3, padding=1)
        self.out_tanh = out_tanh

    def forward(self, x, t):
        h_t = self.time_mlp(sinusoidal_time_embedding(t, dim=64))
        h = F.silu(self.c1(x) + self.t1(h_t).view(-1, 32, 1, 1))
        h = F.silu(self.c2(h) + self.t2(h_t).view(-1, 64, 1, 1))
        h = F.silu(self.c3(h))
        h = F.silu(self.c4(h))
        y = self.out(h)
        return torch.tanh(y) if self.out_tanh else y


## Problem 3 — Train an $x_0$ Predictor

Your tasks:
1. Implement `x0_prediction_loss`.
2. Train `x0_model` for a few epochs.
3. Inspect one-shot clean-image predictions at different timesteps.


In [ ]:
# -- Problem 3: x0 prediction objective -------------------------------------

x0_model = TinyDenoiser(out_tanh=True).to(device)
opt_x0 = optim.AdamW(x0_model.parameters(), lr=2e-4, weight_decay=1e-4)

def x0_prediction_loss(model, x0):
    """Sample t and eps, make x_t, predict x0, return MSE loss."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement x0_prediction_loss')
    # -----------------------------------------------------------------------

print('Training x0 predictor ...')
for epoch in range(5):
    x0_model.train()
    total = 0.0
    for imgs, _ in train_dl:
        imgs = imgs.to(device)
        loss = x0_prediction_loss(x0_model, imgs)
        opt_x0.zero_grad(set_to_none=True)
        loss.backward()
        opt_x0.step()
        total += loss.item()
    print(f'  Epoch {epoch+1:2d}  loss={total/len(train_dl):.5f}')


In [ ]:
# -- Test P3 -----------------------------------------------------------------
x0_model_tmp = TinyDenoiser(out_tanh=True).to(device)
imgs_b = imgs_test[:8].to(device)
loss_b = x0_prediction_loss(x0_model_tmp, imgs_b)
assert loss_b.ndim == 0, 'P3 FAIL: loss should be a scalar tensor'
assert torch.isfinite(loss_b), 'P3 FAIL: loss should be finite'
print('P3 PASSED')

# Visualize one-shot x0 prediction.
x0_model.eval()
with torch.no_grad():
    rows = []
    for t_val in [10, 100, T]:
        t = torch.full((10,), t_val, device=device, dtype=torch.long)
        x_t = q_sample(imgs_test[:10].to(device), t)
        x0_hat = x0_model(x_t, t)
        rows.append((t_val, x_t.cpu(), x0_hat.cpu()))

show_grid(imgs_test[:10], 'clean x0')
for t_val, x_t, x0_hat in rows:
    show_grid(x_t, f'x_t at t={t_val}')
    show_grid(x0_hat, f'x0 prediction from t={t_val}')


## Problem 4 — Generate with the $x_0$ Predictor

Implement two reverse updates:

1. `x0_fresh_step`: predict $\hat x_0$ and use fresh noise.
2. `x0_coherent_step`: predict $\hat x_0$, infer the current noise direction, and reuse it at timestep $t-1$.


In [ ]:
# -- Problem 4: x0 predictor reverse steps ----------------------------------

@torch.no_grad()
def x0_fresh_step(model, x_t, t):
    """Predict x0 and resample a fresh noise direction for x_{t-1}."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement x0_fresh_step')
    # -----------------------------------------------------------------------

@torch.no_grad()
def x0_coherent_step(model, x_t, t):
    """Predict x0, infer eps_hat from current x_t, and reuse it at t-1."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement x0_coherent_step')
    # -----------------------------------------------------------------------

@torch.no_grad()
def sample_x0_model(model, n=64, mode='coherent'):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    for tt in reversed(range(1, T + 1)):
        t = torch.full((n,), tt, device=device, dtype=torch.long)
        if mode == 'fresh':
            x = x0_fresh_step(model, x, t)
        elif mode == 'coherent':
            x = x0_coherent_step(model, x, t)
        else:
            raise ValueError('mode must be fresh or coherent')
    return x.cpu()

x0_samples_fresh = sample_x0_model(x0_model, n=64, mode='fresh')
x0_samples_coherent = sample_x0_model(x0_model, n=64, mode='coherent')
show_grid(x0_samples_fresh, 'x0 predictor: fresh-noise rollout')
show_grid(x0_samples_coherent, 'x0 predictor: coherent rollout')


In [ ]:
# -- Test P4 -----------------------------------------------------------------
x = torch.randn(8, 1, 28, 28, device=device)
t = torch.full((8,), 50, device=device, dtype=torch.long)
xf = x0_fresh_step(x0_model, x, t)
xc = x0_coherent_step(x0_model, x, t)
assert xf.shape == x.shape and xc.shape == x.shape, 'P4 FAIL: reverse steps should preserve shape'
assert torch.isfinite(xf).all() and torch.isfinite(xc).all(), 'P4 FAIL: reverse step outputs must be finite'
print('P4 PASSED')


---
# Section 2 — Backward Process by Predicting Noise

Now train $\varepsilon_\theta(x_t,t)\approx\varepsilon$. This gives a standardized target at every timestep.


## Problem 5 — Train a Noise Predictor

Your tasks:
1. Implement `eps_prediction_loss`.
2. Implement `x0_from_eps_pred`.
3. Compare one-shot $\hat x_0$ at low and high noise.


In [ ]:
# -- Problem 5: epsilon prediction objective --------------------------------

eps_model = TinyDenoiser(out_tanh=False).to(device)
opt_eps = optim.AdamW(eps_model.parameters(), lr=2e-4, weight_decay=1e-4)

def eps_prediction_loss(model, x0):
    """Sample t and eps, make x_t, predict eps, return MSE loss."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement eps_prediction_loss')
    # -----------------------------------------------------------------------


def x0_from_eps_pred(x_t, t, eps_pred):
    """Convert predicted epsilon into a one-shot x0 estimate."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement x0_from_eps_pred')
    # -----------------------------------------------------------------------

print('Training epsilon predictor ...')
for epoch in range(5):
    eps_model.train()
    total = 0.0
    for imgs, _ in train_dl:
        imgs = imgs.to(device)
        loss = eps_prediction_loss(eps_model, imgs)
        opt_eps.zero_grad(set_to_none=True)
        loss.backward()
        opt_eps.step()
        total += loss.item()
    print(f'  Epoch {epoch+1:2d}  loss={total/len(train_dl):.5f}')


In [ ]:
# -- Test P5 -----------------------------------------------------------------
eps_model_tmp = TinyDenoiser(out_tanh=False).to(device)
imgs_b = imgs_test[:8].to(device)
loss_b = eps_prediction_loss(eps_model_tmp, imgs_b)
assert loss_b.ndim == 0 and torch.isfinite(loss_b), 'P5 FAIL: epsilon loss should be finite scalar'

t = torch.full((8,), 20, device=device, dtype=torch.long)
eps = torch.randn_like(imgs_b)
x_t = q_sample(imgs_b, t, eps)
x0_hat = x0_from_eps_pred(x_t, t, eps)
assert F.mse_loss(x0_hat, imgs_b).item() < 1e-10, 'P5 FAIL: true eps should recover x0'
print('P5 PASSED')

# Visualize one-shot x0 estimates from predicted epsilon.
eps_model.eval()
with torch.no_grad():
    for t_val in [10, 100, T]:
        t = torch.full((10,), t_val, device=device, dtype=torch.long)
        x_t = q_sample(imgs_test[:10].to(device), t)
        eps_hat = eps_model(x_t, t)
        x0_hat = x0_from_eps_pred(x_t, t, eps_hat).clamp(-1, 1)
        show_grid(x_t.cpu(), f'epsilon model: x_t at t={t_val}')
        show_grid(x0_hat.cpu(), f'epsilon model: one-shot x0_hat at t={t_val}')


## Problem 6 — Sampling with Predicted Noise

Implement two samplers:

1. `eps_deterministic_step`: same predicted noise direction, lower noise level.
2. `ddpm_step`: posterior mean plus calibrated variance.


In [ ]:
# -- Problem 6: deterministic epsilon step and DDPM posterior step -----------

alpha_bar_prev = alpha_bar0[:-1]
alpha_bar_t = alpha_bar0[1:]
tilde_betas = betas * (1.0 - alpha_bar_prev) / (1.0 - alpha_bar_t)

@torch.no_grad()
def eps_deterministic_step(model, x_t, t):
    """Use predicted eps to form x0_hat, then lower the same noise direction."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement eps_deterministic_step')
    # -----------------------------------------------------------------------


def reverse_mean_from_eps(x_t, t, eps_pred):
    """DDPM reverse mean mu_theta(x_t,t)."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement reverse_mean_from_eps')
    # -----------------------------------------------------------------------

@torch.no_grad()
def ddpm_step(model, x_t, t):
    """One posterior-based DDPM step x_t -> x_{t-1}."""
    # -- YOUR CODE HERE ------------------------------------------------------

    raise NotImplementedError('Implement ddpm_step')
    # -----------------------------------------------------------------------

@torch.no_grad()
def sample_eps_model(model, n=64, mode='ddpm'):
    model.eval()
    x = torch.randn(n, 1, 28, 28, device=device)
    for tt in reversed(range(1, T + 1)):
        t = torch.full((n,), tt, device=device, dtype=torch.long)
        if mode == 'deterministic':
            x = eps_deterministic_step(model, x, t)
        elif mode == 'ddpm':
            x = ddpm_step(model, x, t)
        else:
            raise ValueError('mode must be deterministic or ddpm')
    return x.cpu()

eps_samples_det = sample_eps_model(eps_model, n=64, mode='deterministic')
eps_samples_ddpm = sample_eps_model(eps_model, n=64, mode='ddpm')
show_grid(eps_samples_det, 'epsilon predictor: deterministic rollout')
show_grid(eps_samples_ddpm, 'epsilon predictor: DDPM posterior rollout')


In [ ]:
# -- Test P6 -----------------------------------------------------------------
x = torch.randn(8, 1, 28, 28, device=device)
t = torch.full((8,), 50, device=device, dtype=torch.long)
xd = eps_deterministic_step(eps_model, x, t)
xp = ddpm_step(eps_model, x, t)
assert xd.shape == x.shape and xp.shape == x.shape, 'P6 FAIL: sampler steps should preserve shape'
assert torch.isfinite(xd).all() and torch.isfinite(xp).all(), 'P6 FAIL: sampler outputs must be finite'

# If eps is zero, reverse_mean_from_eps should still produce a finite tensor.
mu = reverse_mean_from_eps(x, t, torch.zeros_like(x))
assert mu.shape == x.shape and torch.isfinite(mu).all(), 'P6 FAIL: reverse mean invalid'
print('P6 PASSED')


---
## Reflection

Questions for discussion:
1. Which sampler looks best after only a few epochs? Why might that change with longer training?
2. Why is one-shot $\hat x_0$ from high-noise $x_t$ unstable?
3. What changes when we move from the deterministic epsilon sampler to the DDPM posterior sampler?
